# Day 09. Exercise 00
# Regularization

## 0. Imports

In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score
from sklearn.ensemble import RandomForestClassifier
from joblib import dump
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier

## 1. Preprocessing

1. Read the file `dayofweek.csv` that you used in the previous day to a dataframe.
2. Using `train_test_split` with parameters `test_size=0.2`, `random_state=21` get `X_train`, `y_train`, `X_test`, `y_test`. Use the additional parameter `stratify`.

In [26]:

dayofweek_df = pd.read_csv('../../data/dayofweek.csv')

X = dayofweek_df.drop(columns='dayofweek')
y = dayofweek_df['dayofweek']

encoder = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
encoded_text = encoder.fit_transform(X[['uid', 'labname']])

cols = encoder.get_feature_names_out(['uid', 'labname'])
X[cols] = encoded_text

X = X.drop(columns=['uid', 'labname'])
numeric_scaler = StandardScaler()
X[['numTrials', 'hour']] = numeric_scaler.fit_transform(X[['numTrials', 'hour']])

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=21,
    stratify=y
)

## 2. Logreg regularization

### a. Default regularization

1. Train a baseline model with the only parameters `random_state=21`, `fit_intercept=False`.
2. Use stratified K-fold cross-validation with `10` splits to evaluate the accuracy of the model


The result of the code where you trained and evaluated the baseline model should be exactly like this (use `%%time` to get the info about how long it took to run the cell):

```
train -  0.62902   |   valid -  0.59259
train -  0.64633   |   valid -  0.62963
train -  0.63479   |   valid -  0.56296
train -  0.65622   |   valid -  0.61481
train -  0.63397   |   valid -  0.57778
train -  0.64056   |   valid -  0.59259
train -  0.64138   |   valid -  0.65926
train -  0.65952   |   valid -  0.56296
train -  0.64333   |   valid -  0.59701
train -  0.63674   |   valid -  0.62687
Average accuracy on crossval is 0.60165
Std is 0.02943
```

In [27]:
def crossval(model, X_train, y_train, folds=10):
    skf = StratifiedKFold(n_splits=folds)
    train_scores, valid_scores = [], []

    for train_split, test_split in skf.split(X_train, y_train):
        X_tr, X_val = X_train.iloc[train_split], X_train.iloc[test_split]
        y_tr, y_val = y_train.iloc[train_split], y_train.iloc[test_split]

        model.fit(X_tr, y_tr)

        train_score = model.score(X_tr, y_tr)
        valid_score = model.score(X_val, y_val)

        train_scores.append(train_score)
        valid_scores.append(valid_score)

        print(f"train - {train_score:.5f} | valid - {valid_score:.5f}")

    avg_valid_score = np.mean(valid_scores)
    std_valid_score = np.std(valid_scores)

    print(f"Average accuracy on crossval: {avg_valid_score:.5f}")
    print(f"Std: {std_valid_score:.5f}\n")

    return avg_valid_score, std_valid_score


In [28]:
%%time
logreg = LogisticRegression(random_state=21, fit_intercept=False)
crossval(logreg, X_train, y_train)


train - 0.62819 | valid - 0.59259
train - 0.64716 | valid - 0.62963
train - 0.63479 | valid - 0.57037
train - 0.65540 | valid - 0.61481
train - 0.63314 | valid - 0.57778
train - 0.64056 | valid - 0.59259
train - 0.64221 | valid - 0.65926
train - 0.65952 | valid - 0.56296
train - 0.64333 | valid - 0.59701
train - 0.63591 | valid - 0.62687
Average accuracy on crossval: 0.60239
Std: 0.02852

CPU times: user 1.54 s, sys: 146 ms, total: 1.69 s
Wall time: 664 ms


(np.float64(0.6023880597014925), np.float64(0.02852354077568261))

### b. Optimizing regularization parameters

1. In the cells below try different values of penalty: `none`, `l1`, `l2` – you can change the values of solver too.

In [29]:
%%capture --no-stdout --no-display
%%time

no_penalty = LogisticRegression(random_state=21, fit_intercept=False, C=np.inf ,max_iter=1000)
no_penalty_score = crossval(no_penalty, X_train, y_train)

l2 = LogisticRegression(random_state=21, fit_intercept=False, l1_ratio=0 ,max_iter=1000)
l2_score = crossval(l2, X_train, y_train)

l1 = LogisticRegression(random_state=21, fit_intercept=False, l1_ratio=1 ,max_iter=1000, solver='saga')
l1_score  = crossval(l1, X_train, y_train)

train - 0.66612 | valid - 0.63704
train - 0.65622 | valid - 0.65926
train - 0.66694 | valid - 0.57778
train - 0.66529 | valid - 0.62963
train - 0.66694 | valid - 0.61481
train - 0.65870 | valid - 0.57778
train - 0.65128 | valid - 0.69630
train - 0.68425 | valid - 0.61481
train - 0.66392 | valid - 0.62687
train - 0.65733 | valid - 0.60448
Average accuracy on crossval: 0.62388
Std: 0.03392

train - 0.62819 | valid - 0.59259
train - 0.64716 | valid - 0.62963
train - 0.63479 | valid - 0.57037
train - 0.65540 | valid - 0.61481
train - 0.63314 | valid - 0.57778
train - 0.64056 | valid - 0.59259
train - 0.64221 | valid - 0.65926
train - 0.65952 | valid - 0.56296
train - 0.64333 | valid - 0.59701
train - 0.63591 | valid - 0.62687
Average accuracy on crossval: 0.60239
Std: 0.02852

train - 0.63726 | valid - 0.58519
train - 0.64221 | valid - 0.61481
train - 0.62984 | valid - 0.55556
train - 0.64386 | valid - 0.60000
train - 0.63232 | valid - 0.57778
train - 0.63644 | valid - 0.57778
train - 0.63

## 3. SVM regularization

### a. Default regularization

1. Train a baseline model with the only parameters `probability=True`, `kernel='linear'`, `random_state=21`.
2. Use stratified K-fold cross-validation with `10` splits to evaluate the accuracy of the model.
3. The format of the result of the code where you trained and evaluated the baseline model should be similar to what you have got for the logreg.

In [30]:
%%time
svc = SVC(probability=True, kernel='linear', random_state=21)
crossval(svc, X_train, y_train)


train - 0.70486 | valid - 0.65926
train - 0.69662 | valid - 0.75556
train - 0.69415 | valid - 0.62222
train - 0.70239 | valid - 0.65185
train - 0.69085 | valid - 0.65185
train - 0.68920 | valid - 0.64444
train - 0.69250 | valid - 0.72593
train - 0.70074 | valid - 0.62222
train - 0.69605 | valid - 0.61940
train - 0.71087 | valid - 0.63433
Average accuracy on crossval: 0.65871
Std: 0.04359

CPU times: user 3.3 s, sys: 36.2 ms, total: 3.34 s
Wall time: 3.16 s


(np.float64(0.6587064676616916), np.float64(0.043585708770590564))

### b. Optimizing regularization parameters

1. In the cells below try different values of the parameter `C`.

In [31]:
%%time
svc = SVC(C=0.1, probability=True, kernel='linear', random_state=21)
crossval(svc, X_train, y_train)

train - 0.58120 | valid - 0.55556
train - 0.57543 | valid - 0.56296
train - 0.57378 | valid - 0.57037
train - 0.59275 | valid - 0.57037
train - 0.58120 | valid - 0.54815
train - 0.57955 | valid - 0.54815
train - 0.57296 | valid - 0.61481
train - 0.59192 | valid - 0.54815
train - 0.59967 | valid - 0.52985
train - 0.57825 | valid - 0.57463
Average accuracy on crossval: 0.56230
Std: 0.02177

CPU times: user 3.48 s, sys: 9.78 ms, total: 3.49 s
Wall time: 5.58 s


(np.float64(0.5622996130458817), np.float64(0.021770909484078532))

In [32]:
%%time
svc = SVC(C=10, probability=True, kernel='linear', random_state=21)
crossval(svc, X_train, y_train)

train - 0.75021 | valid - 0.72593
train - 0.77741 | valid - 0.82963
train - 0.78566 | valid - 0.68148
train - 0.76834 | valid - 0.73333
train - 0.75185 | valid - 0.77778
train - 0.75598 | valid - 0.68889
train - 0.76257 | valid - 0.74074
train - 0.77411 | valid - 0.68889
train - 0.78254 | valid - 0.71642
train - 0.78418 | valid - 0.69403
Average accuracy on crossval: 0.72771
Std: 0.04417

CPU times: user 4.14 s, sys: 1.9 ms, total: 4.14 s
Wall time: 4.14 s


(np.float64(0.7277114427860697), np.float64(0.04417249739078643))

In [33]:
%%time
svc = SVC(C=100, probability=True, kernel='linear', random_state=21)
crossval(svc, X_train, y_train)

train - 0.78401 | valid - 0.74815
train - 0.79720 | valid - 0.84444
train - 0.80956 | valid - 0.72593
train - 0.79060 | valid - 0.76296
train - 0.79060 | valid - 0.77778
train - 0.79637 | valid - 0.74815
train - 0.78401 | valid - 0.77037
train - 0.80462 | valid - 0.73333
train - 0.79819 | valid - 0.70896
train - 0.79901 | valid - 0.73881
Average accuracy on crossval: 0.75589
Std: 0.03550

CPU times: user 14.3 s, sys: 3.37 ms, total: 14.3 s
Wall time: 14.4 s


(np.float64(0.7558872305140961), np.float64(0.035499198685450664))

## 4. Tree

### a. Default regularization

1. Train a baseline model with the only parameter `max_depth=10` and `random_state=21`.
2. Use stratified K-fold cross-validation with `10` splits to evaluate the accuracy of the model.
3. The format of the result of the code where you trained and evaluated the baseline model should be similar to what you have got for the logreg.

In [34]:
%%time
tree = DecisionTreeClassifier(max_depth=10, random_state=21)
crossval(tree, X_train, y_train)


train - 0.81039 | valid - 0.74074
train - 0.77741 | valid - 0.74074
train - 0.83347 | valid - 0.70370
train - 0.79720 | valid - 0.76296
train - 0.82440 | valid - 0.75556
train - 0.80379 | valid - 0.68889
train - 0.80709 | valid - 0.76296
train - 0.80132 | valid - 0.65926
train - 0.80807 | valid - 0.75373
train - 0.80478 | valid - 0.68657
Average accuracy on crossval: 0.72551
Std: 0.03562

CPU times: user 97.5 ms, sys: 47 μs, total: 97.5 ms
Wall time: 98.8 ms


(np.float64(0.7255113322277501), np.float64(0.03562429627613885))

### b. Optimizing regularization parameters

1. In the cells below try different values of the parameter `max_depth`.
2. As a bonus, play with other regularization parameters trying to find the best combination.

In [35]:
%%time
tree = DecisionTreeClassifier(max_depth=5, random_state=21)
crossval(tree, X_train, y_train)


train - 0.59522 | valid - 0.53333
train - 0.56307 | valid - 0.53333
train - 0.60181 | valid - 0.55556
train - 0.59604 | valid - 0.57037
train - 0.60264 | valid - 0.57778
train - 0.57955 | valid - 0.53333
train - 0.58368 | valid - 0.54815
train - 0.59275 | valid - 0.51111
train - 0.58237 | valid - 0.56716
train - 0.60132 | valid - 0.50000
Average accuracy on crossval: 0.54301
Std: 0.02423

CPU times: user 89.9 ms, sys: 994 μs, total: 90.9 ms
Wall time: 91.4 ms


(np.float64(0.5430127142067441), np.float64(0.024234101817933427))

In [36]:
%%time
tree = DecisionTreeClassifier(max_depth=20, random_state=21)
crossval(tree, X_train, y_train)


train - 0.98846 | valid - 0.86667
train - 0.99011 | valid - 0.91111
train - 0.98681 | valid - 0.85926
train - 0.98763 | valid - 0.91111
train - 0.98928 | valid - 0.88148
train - 0.98186 | valid - 0.85926
train - 0.98846 | valid - 0.91852
train - 0.99176 | valid - 0.89630
train - 0.99094 | valid - 0.88060
train - 0.98847 | valid - 0.88060
Average accuracy on crossval: 0.88649
Std: 0.02075

CPU times: user 104 ms, sys: 0 ns, total: 104 ms
Wall time: 106 ms


(np.float64(0.886489773355445), np.float64(0.020748297522140764))

In [37]:
%%time
tree = DecisionTreeClassifier(max_depth=30, random_state=21)
crossval(tree, X_train, y_train)


train - 1.00000 | valid - 0.85926
train - 1.00000 | valid - 0.91852
train - 1.00000 | valid - 0.86667
train - 1.00000 | valid - 0.91111
train - 1.00000 | valid - 0.88148
train - 1.00000 | valid - 0.85185
train - 1.00000 | valid - 0.92593
train - 1.00000 | valid - 0.88148
train - 1.00000 | valid - 0.88060
train - 1.00000 | valid - 0.88060
Average accuracy on crossval: 0.88575
Std: 0.02374

CPU times: user 104 ms, sys: 1.97 ms, total: 106 ms
Wall time: 105 ms


(np.float64(0.8857490326147042), np.float64(0.023739484392796596))

## 5. Random forest

### a. Default regularization

1. Train a baseline model with the only parameters `n_estimators=50`, `max_depth=14`, `random_state=21`.
2. Use stratified K-fold cross-validation with `10` splits to evaluate the accuracy of the model.
3. The format of the result of the code where you trained and evaluated the baseline model should be similar to what you have got for the logreg.

In [38]:
%%time
forest = RandomForestClassifier(n_estimators=50, max_depth=14, random_state=21)
crossval(forest, X_train, y_train)

train - 0.96455 | valid - 0.88148
train - 0.96208 | valid - 0.91852
train - 0.96785 | valid - 0.86667
train - 0.96455 | valid - 0.89630
train - 0.96538 | valid - 0.91111
train - 0.96538 | valid - 0.88148
train - 0.97115 | valid - 0.91852
train - 0.96867 | valid - 0.85185
train - 0.97364 | valid - 0.88060
train - 0.97941 | valid - 0.86567
Average accuracy on crossval: 0.88722
Std: 0.02204

CPU times: user 1.04 s, sys: 3.91 ms, total: 1.04 s
Wall time: 1.04 s


(np.float64(0.8872194582642343), np.float64(0.022044865936245422))

### b. Optimizing regularization parameters

1. In the new cells try different values of the parameters `max_depth` and `n_estimators`.
2. As a bonus, play with other regularization parameters trying to find the best combination.

In [39]:
%%time
forest = RandomForestClassifier(n_estimators=25, max_depth=14, random_state=21)
crossval(forest, X_train, y_train)

train - 0.96373 | valid - 0.88148
train - 0.94312 | valid - 0.89630
train - 0.95960 | valid - 0.84444
train - 0.95796 | valid - 0.88889
train - 0.96620 | valid - 0.90370
train - 0.96373 | valid - 0.85926
train - 0.96373 | valid - 0.90370
train - 0.95960 | valid - 0.85926
train - 0.96211 | valid - 0.88806
train - 0.96293 | valid - 0.87313
Average accuracy on crossval: 0.87982
Std: 0.01925

CPU times: user 582 ms, sys: 5.96 ms, total: 588 ms
Wall time: 596 ms


(np.float64(0.8798231066887784), np.float64(0.019253168703362075))

In [40]:
%%time
forest = RandomForestClassifier(n_estimators=50, max_depth=7, random_state=21)
crossval(forest, X_train, y_train)

train - 0.73042 | valid - 0.65185
train - 0.70899 | valid - 0.72593
train - 0.72383 | valid - 0.62963
train - 0.72795 | valid - 0.67407
train - 0.73619 | valid - 0.71852
train - 0.75433 | valid - 0.68148
train - 0.73124 | valid - 0.74074
train - 0.73372 | valid - 0.62963
train - 0.74629 | valid - 0.67910
train - 0.72076 | valid - 0.65672
Average accuracy on crossval: 0.67877
Std: 0.03703

CPU times: user 905 ms, sys: 6.91 ms, total: 912 ms
Wall time: 910 ms


(np.float64(0.6787672747374239), np.float64(0.037032436617664886))

In [41]:
%%time
forest = RandomForestClassifier(n_estimators=75, max_depth=25, random_state=21)
crossval(forest, X_train, y_train)

train - 0.99918 | valid - 0.90370
train - 0.99918 | valid - 0.94815
train - 0.99918 | valid - 0.89630
train - 1.00000 | valid - 0.94074
train - 0.99835 | valid - 0.91852
train - 0.99918 | valid - 0.89630
train - 0.99918 | valid - 0.92593
train - 0.99918 | valid - 0.88889
train - 1.00000 | valid - 0.92537
train - 1.00000 | valid - 0.91045
Average accuracy on crossval: 0.91543
Std: 0.01878

CPU times: user 1.69 s, sys: 7.88 ms, total: 1.7 s
Wall time: 1.7 s


(np.float64(0.9154339414040906), np.float64(0.018784665368057226))

## 6. Predictions

1. Choose the best model and use it to make predictions for the test dataset.
2. Calculate the final accuracy.
3. Analyze: for which weekday your model makes the most errors (in % of the total number of samples of that class in your test dataset).
4. Save the model.

In [42]:
best_model = RandomForestClassifier(n_estimators=75, max_depth=25, random_state=21)
best_model.fit(X_train, y_train)
best_predict = best_model.predict(X_test)
accuracy_score(y_test, best_predict)

0.9260355029585798

In [43]:

df_pred = pd.DataFrame({
    'target': np.array(y_test),
    'prediction': best_predict
})

df_pred['mistake'] = df_pred['prediction'] != df_pred['target']

res = df_pred.groupby('target')['mistake'].mean().sort_values(ascending=False)  * 100
res

target
0    25.925926
4    14.285714
5     9.259259
1     9.090909
2     6.666667
3     2.500000
6     1.408451
Name: mistake, dtype: float64

Monday is the day with the most mistakes.

In [44]:
dump(best_model, '00_regularization.joblib')

['00_regularization.joblib']